# HRRR weather for a small area

HRRR (High-Resolution Rapid Refresh) is NOAA's 3 km weather model for the
lower 48 states. It runs every hour, takes in radar data, and resolves the
terrain-driven winds, thunderstorm outflows and humidity swings that
drive fire behavior. It also carries fields no Earth Engine dataset has,
such as near-surface wildfire smoke.

It is **not in the Earth Engine catalog**. NOAA publishes it as GRIB2 on
Google Cloud (`gs://high-resolution-rapid-refresh`, 2014 to now), which
Earth Engine cannot read. `geeViz.hrrr.getHRRR` bridges the two:

1. works out which HRRR runs and forecast hours cover your window,
2. downloads **only the fields you ask for**, using each file's byte index
   (about 1.5 MB per field per hour instead of a 160 MB file),
3. crops them to **your region**, converts units, rotates the winds to
   true north, and writes one small Cloud Optimized GeoTIFF per hour to a
   bucket you own,
4. returns an `ee.ImageCollection` that Earth Engine reads from those
   files directly. Nothing is ingested, so there is no wait.

How it compares with what is already in Earth Engine:

| Dataset | Grid | Kind | Use it for |
|---|---|---|---|
| HRRR (this notebook) | 3 km, hourly | analysis + forecast to 48 h | the next two days; smoke; storm-scale wind |
| `NOAA/NWS/RTMA` | 2.5 km, hourly | analysis (observation-corrected) | looking back over days to years |
| `NOAA/GFS0P25` (via `geeViz.weather`) | ~28 km | forecast to 16 days | anything past 48 hours |

You need a Cloud Storage bucket you can write to and Earth Engine can
read. Set it below or in the `GEEVIZ_HRRR_BUCKET` environment variable.

## Setup

In [1]:
import datetime
import math
import os

import pandas as pd

try:
    import geeViz.geeView as gv
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geeViz'])
    import geeViz.geeView as gv

import geeViz.weather as wx
from geeViz.hrrr import getHRRR

ee = gv.ee
Map = gv.Map
Map.clearMap()
pd.set_option('display.width', 200)

# A bucket you can write to and Earth Engine can read.
HRRR_BUCKET = os.environ.get('GEEVIZ_HRRR_BUCKET', 'gs://your-bucket/hrrr')

# A typical question area: one county.
county = ee.FeatureCollection('TIGER/2018/Counties').filter(ee.Filter.eq('GEOID', '08013'))
print('Study area:', county.first().get('NAME').getInfo(), 'County, Colorado')

[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:8889/ee-api (attached to existing detached proxy, tenant_header=X-geeViz-Creds)


geeViz: Earth Engine ready (project='geeviz-geo-agent', source=eeauth-proxy)


Study area: Boulder County, Colorado


## Fetch one afternoon

Six hours on 2026-08-07, noon to 6 pm Mountain time (18:00 to 24:00 UTC).
The window works like `filterDate`: the start is included, the end is not.

`verbose=True` (the default) prints the plan before anything is
downloaded (which hours, from which runs, and how many are already
cached), then shows a status bar. While it runs, the bar counts how many
hours are fetching, decoding and uploading, the megabytes moved, and the
last hour finished.

In [2]:
VARIABLES = ['temperature_2m', 'relative_humidity_2m', 'wind', 'wind_gust',
             'precipitation', 'smoke_near_surface']

hrrr = getHRRR(county, '2026-08-07T18:00', '2026-08-08T00:00',
               variables=VARIABLES, bucket=HRRR_BUCKET)

HRRR: 6 hour(s), 2026-08-07 18:00 to 2026-08-07 23:00 UTC, every 1 h
  region   -105.694, 39.913, -105.053, 40.263 (W, S, E, N)
  bands    temperature_2m, relative_humidity_2m, u, v, wind_gust, precipitation, smoke_near_surface
  newest   2026-11-02 19z run published
  past     6 hour(s) from each hour's own analysis (f00)


  cache    0 of 6 hour(s) already prepared; 6 to fetch


HRRR [####################] 6/6 hours, 9.9 s | idle | 57.9 MB fetched, 0.12 MB written | last done: 08-07 23Z (analysis, 23z run f00), 20 KB

Done in 21.3 s: 6 hour(s) prepared, 0 from cache; fetched 57.9 MB of GRIB2, wrote 0.12 MB.


## What came back

One image per hour. `system:time_start` is the **valid** time, the hour
the weather is for, not the time the model ran. Each image says where it
came from: past hours are HRRR's own analysis (forecast hour 0 of the run
that started that hour).

In [3]:
def describe(collection):
    props = ee.List(['system:time_start', 'run_time', 'lead_hours', 'hrrr_source'])
    rows = collection.toList(collection.size()).map(
        lambda i: ee.Image(i).toDictionary(props).set('bands', ee.Image(i).bandNames()))
    df = pd.DataFrame(rows.getInfo())
    utc = lambda ms: pd.to_datetime(ms, unit='ms', utc=True)
    return pd.DataFrame({
        'valid (UTC)': utc(df['system:time_start']).dt.strftime('%Y-%m-%d %H:%M'),
        'valid (Mountain)': utc(df['system:time_start']).dt.tz_convert('America/Denver').dt.strftime('%m-%d %H:%M'),
        'run (UTC)': utc(df['run_time']).dt.strftime('%m-%d %H') + 'z',
        'lead (h)': df['lead_hours'],
        'source': df['hrrr_source'],
    })

print('Bands:', ', '.join(hrrr.first().bandNames().getInfo()))
print('Units:', hrrr.first().get('wx_units').getInfo())
describe(hrrr)

Bands: temperature_2m, relative_humidity_2m, u, v, wind_gust, precipitation, smoke_near_surface


Units: {'precipitation': 'mm/hr', 'relative_humidity_2m': '%', 'smoke_near_surface': 'ug/m3', 'temperature_2m': 'C', 'u': 'm/s', 'v': 'm/s', 'wind_gust': 'm/s'}


,valid (UTC),valid (Mountain),run (UTC),lead (h),source
0,2026-08-07 18:00,08-07 12:00,08-07 18z,0,analysis
1,2026-08-07 19:00,08-07 13:00,08-07 19z,0,analysis
2,2026-08-07 20:00,08-07 14:00,08-07 20z,0,analysis
3,2026-08-07 21:00,08-07 15:00,08-07 21z,0,analysis
4,2026-08-07 22:00,08-07 16:00,08-07 22z,0,analysis
5,2026-08-07 23:00,08-07 17:00,08-07 23z,0,analysis


## Fire weather over the county, hour by hour

The Fosberg Fire Weather Index combines temperature, humidity and wind
speed into one 0 to 100 number; above about 50 is critical. It is
computed per 3 km cell and then summarized, because the index of an
average day is not the average of the index.

Wind direction is the bearing the wind blows **from**, taken from the
county-mean wind vector.

In [4]:
def fosberg(img):
    t_f = img.select('temperature_2m').multiply(9 / 5).add(32)
    rh = img.select('relative_humidity_2m')
    mph = img.select('u').hypot(img.select('v')).multiply(2.23694)
    # Equilibrium moisture content (Simard 1968), in three humidity ranges.
    m = (ee.Image(0)
         .where(rh.lt(10), rh.multiply(0.281073).subtract(rh.multiply(t_f).multiply(0.000578)).add(0.03229))
         .where(rh.gte(10).And(rh.lte(50)), rh.multiply(0.160107).subtract(t_f.multiply(0.01478)).add(2.22749))
         .where(rh.gt(50), rh.pow(2).multiply(0.005565).subtract(rh.multiply(t_f).multiply(0.00035))
                .subtract(rh.multiply(0.483199)).add(21.0606)))
    r = m.divide(30)
    eta = ee.Image(1).subtract(r.multiply(2)).add(r.pow(2).multiply(1.5)).subtract(r.pow(3).multiply(0.5))
    return eta.multiply(mph.pow(2).add(1).sqrt()).divide(0.3002).min(100).rename('fosberg')

def summarize(img):
    stack = img.addBands(fosberg(img)).addBands(img.select('u').hypot(img.select('v')).rename('speed'))
    mean = stack.reduceRegion(ee.Reducer.mean(), county.geometry(), 3000)
    peak = stack.select(['wind_gust', 'fosberg']).reduceRegion(ee.Reducer.max(), county.geometry(), 3000)
    return ee.Feature(None, mean).set('gust_max', peak.get('wind_gust'),
                                      'fosberg_max', peak.get('fosberg'),
                                      'time', img.get('system:time_start'))

rows = pd.DataFrame([f['properties'] for f in hrrr.map(summarize).getInfo()['features']])
table = pd.DataFrame({
    'Mountain time': pd.to_datetime(rows['time'], unit='ms', utc=True).dt.tz_convert('America/Denver').dt.strftime('%m-%d %H:%M'),
    'temp (C)': rows['temperature_2m'].round(1),
    'RH (%)': rows['relative_humidity_2m'].round(0),
    'wind (m/s)': rows['speed'].round(1),
    'from (deg)': [round((math.degrees(math.atan2(-u, -v))) % 360) for u, v in zip(rows['u'], rows['v'])],
    'gust max (m/s)': rows['gust_max'].round(1),
    'precip (mm/hr)': rows['precipitation'].round(2),
    'smoke (ug/m3)': rows['smoke_near_surface'].round(1),
    'Fosberg mean': rows['fosberg'].round(0),
    'Fosberg max': rows['fosberg_max'].round(0),
})
table

,Mountain time,temp (C),RH (%),wind (m/s),from (deg),gust max (m/s),precip (mm/hr),smoke (ug/m3),Fosberg mean,Fosberg max
0,08-07 12:00,26.3,16.0,3.2,58,6.8,0,13.7,19.0,36.0
1,08-07 13:00,27.4,16.0,3.3,66,6.8,0,13.3,20.0,35.0
2,08-07 14:00,28.5,14.0,3.4,60,7.4,0,12.0,21.0,32.0
3,08-07 15:00,28.6,14.0,3.2,80,8.0,0,11.9,19.0,40.0
4,08-07 16:00,28.7,13.0,3.8,72,9.7,0,11.0,23.0,45.0
5,08-07 17:00,28.8,12.0,3.2,22,10.8,0,10.5,20.0,68.0


## Ask again: the cache

Files are named by run, forecast hour, variables and region, so the same
request reads what is already in the bucket. Nothing is downloaded and
the status bar does not appear.

In [5]:
hrrr_again = getHRRR(county, '2026-08-07T18:00', '2026-08-08T00:00',
                     variables=VARIABLES, bucket=HRRR_BUCKET)

HRRR: 6 hour(s), 2026-08-07 18:00 to 2026-08-07 23:00 UTC, every 1 h
  region   -105.694, 39.913, -105.053, 40.263 (W, S, E, N)
  bands    temperature_2m, relative_humidity_2m, u, v, wind_gust, precipitation, smoke_near_surface
  newest   2026-11-02 19z run published
  past     6 hour(s) from each hour's own analysis (f00)


  cache    6 of 6 hour(s) already prepared; 0 to fetch
Done in 2.3 s: 0 hour(s) prepared, 6 from cache; fetched 0.0 MB of GRIB2, wrote 0.00 MB.


## On the map

Temperature as a time lapse, and the wind as an animated time lapse
(speed raster plus particles). `addWindTimeLapse` reads the `u` and `v`
bands, which `getHRRR` has already rotated from HRRR's grid to true
east and north.

In [6]:
Map.clearMap()
TL = {'dateFormat': 'YY-MM-dd HH', 'advanceInterval': 'hour'}
Map.addTimeLapse(hrrr.select('temperature_2m'),
                 {'min': 15, 'max': 35, 'palette': list(wx.TEMPERATURE_PALETTE),
                  'yLabel': 'Temperature (C)', **TL},
                 'HRRR 2 m temperature (UTC)')
Map.addWindTimeLapse(hrrr.select(['u', 'v']), {'units': 'mi/hr'},
                     'HRRR 10 m wind (UTC)', **TL)
Map.addLayer(county.style(color='000', fillColor='00000000', width=2), {}, 'Boulder County')
Map.centerObject(county, 10)
Map.turnOnInspector()
Map.view()

Adding layer: HRRR 2 m temperature (UTC)
Adding layer: HRRR 10 m wind (UTC)
Adding layer: Boulder County


Starting webmap
Using eeCreds proxy at http://127.0.0.1:8889/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:8889/geeView/?v=1793649332856


## The next few hours

Hours still ahead come from **one** run: the newest one that reaches the
end of the window. Most runs reach 18 hours; the 00, 06, 12 and 18 UTC
runs reach 48. Hours before that run's start come from analyses, so a
window that starts in the past and ends in the future has both, tagged in
`source`.

In [7]:
now = datetime.datetime.now(datetime.timezone.utc).replace(minute=0, second=0, microsecond=0)
ahead = getHRRR(county, now - datetime.timedelta(hours=2), now + datetime.timedelta(hours=4),
                variables=['temperature_2m', 'relative_humidity_2m', 'wind'], bucket=HRRR_BUCKET)
describe(ahead)

HRRR: 6 hour(s), 2026-11-02 17:00 to 2026-11-02 22:00 UTC, every 1 h
  region   -105.694, 39.913, -105.053, 40.263 (W, S, E, N)
  bands    temperature_2m, relative_humidity_2m, u, v
  newest   2026-11-02 19z run published
  past     2 hour(s) from each hour's own analysis (f00)
  ahead    4 hour(s) from the 2026-11-02 18z run, f01 to f04
  cache    0 of 6 hour(s) already prepared; 6 to fetch


HRRR [####################] 6/6 hours, 5.0 s | idle | 44.8 MB fetched, 0.08 MB written | last done: 11-02 18Z (analysis, 18z run f00), 13 KB

Done in 6.3 s: 6 hour(s) prepared, 0 from cache; fetched 44.8 MB of GRIB2, wrote 0.08 MB.


,valid (UTC),valid (Mountain),run (UTC),lead (h),source
0,2026-11-02 17:00,11-02 10:00,11-02 17z,0,analysis
1,2026-11-02 18:00,11-02 11:00,11-02 18z,0,analysis
2,2026-11-02 19:00,11-02 12:00,11-02 18z,1,forecast
3,2026-11-02 20:00,11-02 13:00,11-02 18z,2,forecast
4,2026-11-02 21:00,11-02 14:00,11-02 18z,3,forecast
5,2026-11-02 22:00,11-02 15:00,11-02 18z,4,forecast


## Guards

`getHRRR` refuses rather than quietly returning less than you asked for:

* **Too many hours.** Each hour takes about 12 s of download and
  conversion whatever the area (8 run at once), so windows over `max_hours` (72 by default) raise before downloading
  anything. Use `hourStep` to thin the window, raise `max_hours`, or use
  RTMA for a long look-back.
* **Too far ahead.** HRRR stops at 48 hours. Past that it raises and
  says where coverage ends.

In [8]:
for label, start, end in [
    ('a week, hourly', '2026-08-01', '2026-08-08'),
    ('four days ahead', now + datetime.timedelta(days=3), now + datetime.timedelta(days=4)),
]:
    try:
        getHRRR(county, start, end, bucket=HRRR_BUCKET, verbose=False)
    except ValueError as e:
        print(f'{label}:\n  {e}\n')

a week, hourly:
  168 hours requested; max_hours is 72. That would take about 4 minutes. Use hourStep to thin the window, raise max_hours, or, for a long look-back, use NOAA/NWS/RTMA, which is already in Earth Engine.



four days ahead:
  HRRR does not reach 2026-11-06 18:00 UTC yet. The newest published run is 2026-11-02 19z; the newest long run reaches 2026-11-04 18:00 UTC at most (48 hours from 00/06/12/18z, 18 from other runs). End the window earlier, or use geeViz.weather.getForecastData(..., model='gfs'), which reaches 16 days.



## Notes

* **Winds are rotated to true north.** HRRR publishes u and v relative to
  its Lambert grid, and GDAL does not correct for it. Unrotated, the
  direction is off by up to about 18 degrees at the edges of the grid.
  Checked against ERA5 over open ocean on 2026-08-06 (8 hours each):
  HRRR's direction minus ERA5's was +25.5 degrees off Oregon and -18.5 in
  the Gulf of Maine before rotation, and +7.7 and -0.3 after.
* **Units** follow `geeViz.weather`: Celsius, percent, m/s, mm/hr, and
  ug/m3 for smoke. Each image carries them in `wx_units`.
* **Precipitation** is the hour ending at the valid time. Past hours take
  it from the previous run's first hour, because a run's hour 0 has none.
* **Coverage** is the lower 48 states (HRRR's CONUS domain).
* **Bucket access.** Files are written with your Google Cloud credentials
  and read by Earth Engine with the account it runs as. If those differ,
  give that account read access to the bucket.